我要看看创新高后，多日后结果是什么

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
# 【关键两行】——必须同时设置！
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'PingFang SC', 'Heiti TC', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题[1][2][3][4][5][6][7][8][9][10]

import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/聚宽数据')
import datasource 
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.getAStockCodes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:4939,第一支股票:000001.XSHE


index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-07-17      0  6.65  6.65  6.59   6.63  3.263970e+08   
            2018-07-18      1  6.65  6.73  6.61   6.62  4.606974e+08   
            2018-07-19      2  6.65  6.70  6.61   6.64  4.036743e+08   
            2018-07-20      3  6.64  7.00  6.55   6.93  1.576844e+09   
            2018-07-23      4  6.87  7.19  6.84   7.16  1.646028e+09   

                             volume  
code        date                     
000001.XSHE 2018-07-17   49362461.0  
            2018-07-18   69061931.0  
            2018-07-19   60674284.0  
            2018-07-20  230701981.0  
            2018-07-23  234306736.0

# 先看看一字涨停是前面的新高，会发生什么

In [7]:
N = 20 # 1个月当作20天的
dt[f'pre_max_{N}'] = dt['high'].rolling(window=N).max().shift(1) # rolling是滑动窗口，shift下移一天。
dt[f'pre_min_{N}'] = dt['low'].rolling(window=N).min().shift(1) # rolling是滑动窗口，shift下移一天。
dt['change_percent_5'] = (dt['close'].shift(-5)-dt['close'])/dt['close']*100
dt.head(25)

index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-07-17      0  6.65  6.65  6.59   6.63  3.263970e+08   
            2018-07-18      1  6.65  6.73  6.61   6.62  4.606974e+08   
            2018-07-19      2  6.65  6.70  6.61   6.64  4.036743e+08   
            2018-07-20      3  6.64  7.00  6.55   6.93  1.576844e+09   
            2018-07-23      4  6.87  7.19  6.84   7.16  1.646028e+09   
            2018-07-24      5  7.14  7.29  7.09   7.15  1.348747e+09   
            2018-07-25      6  7.18  7.20  7.09   7.13  5.757522e+08   
            2018-07-26      7  7.15  7.21  7.00   7.03  7.784298e+08   
            2018-07-27      8  7.07  7.12  6.98   7.03  4.775274e+08   
            2018-07-30      9  7.09  7.22  7.04   7.14  6.617675e+08   
            2018-07-31     10  7.13  7.22  7.03   7.16  6.744847e+08   
            2018-08-01     11  7.16  7.22  6.93   6.96  7.592822e+08   
            2018-08-02     12  6.94  6.96  6.75   6.80  8.392723e+08   
            2018-08-03     13  6.79  6.92  6.78   6.78  4.286415e+08   
            2018-08-06     14  6.80  6.93  6.76   6.80  4.979067e+08   
            2018-08-07     15  6.81  6.97  6.75   6.97  6.243597e+08   
            2018-08-08     16  6.97  6.97  6.87   6.88  5.275270e+08   
            2018-08-09     17  6.87  7.11  6.86   7.06  7.464698e+08   
            2018-08-10     18  7.07  7.09  6.97   7.02  3.913663e+08   
            2018-08-13     19  6.93  6.97  6.77   6.88  7.369478e+08   
            2018-08-14     20  6.87  6.87  6.80   6.84  3.863346e+08   
            2018-08-15     21  6.86  6.86  6.65   6.68  5.947595e+08   
            2018-08-16     22  6.64  6.76  6.57   6.68  5.595231e+08   
            2018-08-17     23  6.75  6.80  6.68   6.70  5.766833e+08   
            2018-08-20     24  6.77  6.86  6.74   6.84  6.099347e+08   

                             volume  pre_max_20  pre_min_20  change_percent_5  \
code        date                                                                
000001.XSHE 2018-07-17   49362461.0         NaN         NaN          7.843137   
            2018-07-18   69061931.0         NaN         NaN          7.703927   
            2018-07-19   60674284.0         NaN         NaN          5.873494   
            2018-07-20  230701981.0         NaN         NaN          1.443001   
            2018-07-23  234306736.0         NaN         NaN         -0.279330   
            2018-07-24  187516870.0         NaN         NaN          0.139860   
            2018-07-25   80762227.0         NaN         NaN         -2.384292   
            2018-07-26  109593486.0         NaN         NaN         -3.271693   
            2018-07-27   67743332.0         NaN         NaN         -3.556188   
            2018-07-30   92895595.0         NaN         NaN         -4.761905   
            2018-07-31   94705784.0         NaN         NaN         -2.653631   
            2018-08-01  107058471.0         NaN         NaN         -1.149425   
            2018-08-02  122486968.0         NaN         NaN          3.823529   
            2018-08-03   62669820.0         NaN         NaN          3.539823   
            2018-08-06   72856906.0         NaN         NaN          1.176471   
            2018-08-07   90796357.0         NaN         NaN         -1.865136   
            2018-08-08   76137127.0         NaN         NaN         -2.906977   
            2018-08-09  106098736.0         NaN         NaN         -5.382436   
            2018-08-10   55781646.0         NaN         NaN         -4.558405   
            2018-08-13  107638349.0         NaN         NaN         -0.581395   
            2018-08-14   56580164.0        7.29        6.55          2.631579   
            2018-08-15   88168777.0        7.29        6.55          5.688623   
            2018-08-16   83938117.0        7.29        6.55          6.586826   
            2018-08-17   85411537.0        7.29       

In [8]:
# 我这里看看是否有一字涨停的，
dt2 = dt.loc[ (dt['close'] >  dt['pre_max_20']) & (dt[f'pre_max_{N}']/dt[f'pre_min_{N}'] < 1.2), :]
print(len(dt2))

100


In [9]:
_avg = dt2['change_percent_5'].mean()
_min = dt2['change_percent_5'].min()
_median = dt2['change_percent_5'].median()
print(f'平均:{_avg},最低:{_min}, 中位值:{_median}')

平均:-0.14021565502168248,最低:-6.785714285714289, 中位值:-0.04638218923933111


In [10]:
# 我想看看2个月，3个月的
for days  in [20, 40,60,80,100]:
    dt[f'pre_max_{days}'] = dt['high'].rolling(window=days).max().shift(1) # rolling是滑动窗口，shift下移一天。
    dt2 = dt.loc[ dt['close'] >  dt[f'pre_max_{days}'], :]
    _avg = dt2['change_percent_5'].mean()
    _min = dt2['change_percent_5'].min()
    _median = dt2['change_percent_5'].median()
    _max  = dt2['change_percent_5'].max()
    print(f'天数:{days:3d},最高:{_max:.1f},平均:{_avg:.1f},最低:{_min:.1f}, 中位值:{_median:.1f}')
    


天数: 20,最高:11.4,平均:-0.4,最低:-6.8, 中位值:-0.3
天数: 40,最高:9.9,平均:-0.2,最低:-6.8, 中位值:-0.1
天数: 60,最高:9.9,平均:0.1,最低:-6.6, 中位值:0.3
天数: 80,最高:9.9,平均:0.2,最低:-6.3, 中位值:0.5
天数:100,最高:9.9,平均:0.3,最低:-6.3, 中位值:0.6


In [11]:
# 我这里看看全部股票的
dict_result = {
    'code':[]
}
for days  in [20, 40,60,80,100]:
    dict_result[f'{days}最高'] = []
    dict_result[f'{days}最低'] = []
    dict_result[f'{days}平均'] = []
    dict_result[f'{days}中位值'] = []

for i in tqdm(range(len(codes))):
    code = codes[i]
    dt = datasource.getData(code)
    dt['change_percent_5'] = (dt['close'].shift(-5)-dt['close'])/dt['close']*100
    dict_result['code'].append(code) # 先将股票代码放进去
    for days  in [20, 40,60,80,100]:
        dt[f'pre_max_{days}'] = dt['high'].rolling(window=days).max().shift(1) # rolling是滑动窗口，shift下移一天。
        dt[f'pre_min_{N}'] = dt['high'].rolling(window=N).min().shift(1)
        dt2 = dt.loc[ (dt['close'] >  dt[f'pre_max_{days}']) & (dt[f'pre_max_{N}']/dt[f'pre_min_{N}'] < 1.2), :]
        _avg = dt2['change_percent_5'].mean()
        _min = dt2['change_percent_5'].min()
        _median = dt2['change_percent_5'].median()
        _max  = dt2['change_percent_5'].max()
        dict_result[f'{days}最高'].append(_max)
        dict_result[f'{days}最低'].append(_min)
        dict_result[f'{days}平均'].append(_avg)
        dict_result[f'{days}中位值'].append(_median)
# 到这里得到了
df  = pd.DataFrame(dict_result)
df.to_excel('传新高所有股票.xlsx')

100%|██████████████████████████████████████████████████████████████████████████████| 4939/4939 [02:41<00:00, 30.61it/s]


创新高后好像并没有继续。